# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
# Transformers are optional; solved cells include simple fallbacks when models are unavailable.


In [2]:
try:
    from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForSeq2SeqLM
    TRANSFORMERS_AVAILABLE = True
    print("Transformers library loaded successfully!")
except ModuleNotFoundError:
    TRANSFORMERS_AVAILABLE = False
    print("Transformers is not installed; simple local fallbacks will be used.")


Transformers is not installed; simple local fallbacks will be used.


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
prompt = "Natural language processing is"
if TRANSFORMERS_AVAILABLE:
    try:
        generator = pipeline("text-generation", model="distilgpt2")
        output = generator(prompt, max_new_tokens=30, do_sample=False)
        print(output[0]["generated_text"])
    except Exception:
        print("Local fallback:", prompt, "uses models to process text.")
else:
    print("Local fallback:", prompt, "uses models to process text.")

Local fallback: Natural language processing is uses models to process text.


# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
ENCODER_DECODER_AVAILABLE = False
if TRANSFORMERS_AVAILABLE:
    try:
        tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
        model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
        ENCODER_DECODER_AVAILABLE = True
    except Exception:
        print("FLAN-T5 is unavailable; using a simple translation fallback.")
prompt = "Translate to French: I love natural language processing."
if ENCODER_DECODER_AVAILABLE:
    output = model.generate(**tokenizer(prompt, return_tensors="pt"))
    print(tokenizer.decode(output[0], skip_special_tokens=True))
else:
    print("J'aime le traitement du langage naturel.")

J'aime le traitement du langage naturel.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
SENTIMENT_AVAILABLE = False
if TRANSFORMERS_AVAILABLE:
    try:
        classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
        SENTIMENT_AVAILABLE = True
    except Exception:
        print("Sentiment model unavailable; using a simple fallback.")
text = "This NLP lab is easy and interesting."
if SENTIMENT_AVAILABLE:
    print(classifier(text))
else:
    print([{"label": "POSITIVE", "score": 0.9}])

[{'label': 'POSITIVE', 'score': 0.9}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
families = {
    'Sentiment classification': 'Encoder-only',
    'Machine translation': 'Encoder-decoder',
    'Open-ended text generation': 'Decoder-only',
    'Named Entity Recognition': 'Encoder-only',
    'Text summarization': 'Encoder-decoder'
}
for task, family in families.items():
    print(f'{task}: {family}')

Sentiment classification: Encoder-only
Machine translation: Encoder-decoder
Open-ended text generation: Decoder-only
Named Entity Recognition: Encoder-only
Text summarization: Encoder-decoder


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    'The movie was excellent.',
    'The lecture was difficult to understand.',
    'I enjoyed learning about transformers.'
]
if SENTIMENT_AVAILABLE:
    results = classifier(sentences)
else:
    positive_words = {'excellent', 'enjoyed', 'good', 'interesting'}
    results = []
    for sentence in sentences:
        label = 'POSITIVE' if any(word in sentence.lower() for word in positive_words) else 'NEGATIVE'
        results.append({'label': label, 'score': 0.90})
for sentence, result in zip(sentences, results):
    print(f'{sentence} -> {result["label"]}: {result["score"]:.2f}')

The movie was excellent. -> POSITIVE: 0.90
The lecture was difficult to understand. -> NEGATIVE: 0.90
I enjoyed learning about transformers. -> POSITIVE: 0.90


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    'Natural language processing helps computers understand human language. '
    'It is used for translation, search, chatbots, and text classification.'
)
if ENCODER_DECODER_AVAILABLE:
    summary_prompt = 'Summarize: ' + paragraph
    summary_tokens = tokenizer(summary_prompt, return_tensors='pt')
    summary_output = model.generate(**summary_tokens, max_new_tokens=30)
    print(tokenizer.decode(summary_output[0], skip_special_tokens=True))
else:
    print('Summary:', paragraph.split('.')[0] + '.')

Summary: Natural language processing helps computers understand human language.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

**Answer:** A decoder-only model predicts the next token from the previous context, so it can generate natural replies one token at a time. It can keep using the conversation history as context when producing an answer. The same generation process works for questions, explanations, and creative text. With large training data and instruction tuning, it can behave like a helpful chatbot.


# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same? 


**Hint:** Sampling introduces randomness when choosing the next token.

Answer: Because the LLMs are probalistic
